# Análisis de Patrones de Actualización - Contraloría Panamá

Este análisis examina los patrones de actualización de la fuente de datos para identificar:
- Frecuencia de actualizaciones
- Día de la semana preferido
- Hora estimada de actualización
- Diferencias entre instituciones y estados

In [0]:
%sql 
select distinct  institution_sp from contraloria.employee_payroll.silver_employee_payroll_latest_snapshot LEFT JOIN
  (select DISTINCT  institution_name_spanish from contraloria.reference_and_audit.reference_institution_names) ON institution_name_spanish = institution_en where institution_name_spanish IS NULL

In [0]:
%sql select * from contraloria.reference_and_audit.reference_institution_names where institution_name_spanish IS NULL

In [0]:
# 1. Explorar fechas únicas de consulta por institución y estado
from pyspark.sql import functions as F
from datetime import datetime, timedelta
import pandas as pd

# Leer datos de la tabla bronze
df_raw = spark.table("contraloria.employee_payroll.bronze_contraloria_employees_raw")

# Obtener combinaciones únicas de institución, estado y fecha de consulta
update_dates = df_raw
.select(
    F.col("institucion"),
    F.col("estado"),
    F.col("fecha_consulta"),
    F.date_format("fecha_consulta", "yyyy-MM-dd").alias("fecha"),
    F.date_format("fecha_consulta", "EEEE").alias("dia_semana"),
    F.date_format("fecha_consulta", "HH:mm:ss").alias("hora"),
    F.dayofweek("fecha_consulta").alias("dia_numero")
).distinct().orderBy(F.desc("fecha_consulta"))

print(f"Total de combinaciones únicas (institución + estado + fecha): {update_dates.count()}")
print("\n20 combinaciones más recientes:")
display(update_dates.limit(20))

In [0]:
# 2. Análisis de frecuencia entre actualizaciones
# Obtener fechas únicas de consulta ordenadas
update_dates_ordered = update_dates.select("fecha").distinct().orderBy("fecha")
update_dates_pd = update_dates_ordered.toPandas()


update_dates_pd['fecha'] = pd.to_datetime(update_dates_pd['fecha'])
update_dates_pd['days_between_updates'] = update_dates_pd['fecha'].diff().dt.days



q1 = update_dates_pd['days_between_updates'].quantile(0.25)
q3 = update_dates_pd['days_between_updates'].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

update_dates_pd["is_outlier_iqr"] = update_dates_pd['days_between_updates'] > upper_bound

# versión limpia
df_clean = update_dates_pd[~update_dates_pd["is_outlier_iqr"]].copy()

print(f"""
    median: {update_dates_pd["days_between_updates"].median()}
    mean_all: {update_dates_pd["days_between_updates"].mean()}
    median_clean: {df_clean["days_between_updates"].median()}
    mean_clean: {df_clean["days_between_updates"].mean()}
    upper_bound: {upper_bound}
    outlier_count: {int(update_dates_pd["is_outlier_iqr"].sum())}
    """
)




In [0]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Histograma de todas las fechas
axes[0].hist(update_dates_pd['days_between_updates'].dropna(), bins=20, alpha=0.7, color='skyblue')
axes[0].set_xlabel('Días entre actualizaciones')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Todas las fechas')
axes[0].grid(True)

# Histograma sin outliers
axes[1].hist(df_clean['days_between_updates'].dropna(), bins=20, alpha=0.7, color='salmon')
axes[1].set_xlabel('Días entre actualizaciones')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Sin outliers')
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [0]:
# 3. Análisis de día de la semana y hora
day_analysis = df_raw.select(
    F.date_format("fecha_consulta", "EEEE").alias("dia_semana"),
    F.dayofweek("fecha_consulta").alias("dia_numero"),
    F.date_format("fecha_consulta", "HH:mm").alias("hora"),
    F.hour("fecha_consulta").alias("hora_numero")
).distinct()

day_counts = day_analysis.groupBy("dia_semana", "dia_numero").count().orderBy("dia_numero")
print("Distribución por día de la semana:")
display(day_counts)

hour_stats = day_analysis.groupBy("hora_numero").count().orderBy("count", ascending=False)
print("\nHoras más comunes de actualización:")
display(hour_stats.limit(10))


In [0]:
# 4. Análisis por institución y estado
# Contar cuántas fechas únicas hay por combinación de institución y estado
unique_dates_per_combo = df_raw.groupBy(
    "institucion",
    "estado"
).agg(
    F.countDistinct("fecha_consulta").alias("fechas_unicas"),
    F.max("fecha_consulta").alias("ultima_actualizacion"),
    F.min("fecha_consulta").alias("primera_actualizacion")
).orderBy(F.desc("fechas_unicas"))

print("Combinaciones de institución y estado con su historial de actualizaciones:")
print(f"Total de combinaciones: {unique_dates_per_combo.count()}")
print("\nPrimeras 30 combinaciones con más variedad de fechas:")
display(unique_dates_per_combo.limit(30))

# Verificar si todas las combinaciones se actualizan al mismo tiempo
same_date_check = df_raw.select("fecha_consulta").distinct().count()
print(f"\nTotal de fechas únicas de consulta en toda la tabla: {same_date_check}")
print("Si este número es pequeño, significa que todas las instituciones/estados se actualizan simultáneamente")

# Análisis de sincronización
print("\n\u00bfHay diferencias en las fechas de actualización por institución/estado?")
combinations_count = unique_dates_per_combo.count()
print(f"Total de combinaciones institución-estado: {combinations_count}")
print(f"Total de fechas únicas de consulta: {same_date_check}")
if same_date_check < combinations_count * 0.1:
    print("\u2192 Las actualizaciones son SINCRONIZADAS: todas las instituciones/estados se actualizan al mismo tiempo")
else:
    print("\u2192 Las actualizaciones son DIFERENCIADAS: diferentes instituciones/estados tienen horarios distintos")

In [0]:
# 5. Predicción de próxima actualización
from datetime import datetime, timedelta

# Obtener la última fecha de consulta
last_update = df_raw.agg(F.max("fecha_consulta")).collect()[0][0]
print(f"Última actualización registrada: {last_update}")
print(f"Fecha actual: {datetime.now()}")

if len(update_dates_pd) > 1 and update_dates_pd['dias_desde_anterior'].notna().sum() > 0:
    avg_days = update_dates_pd['dias_desde_anterior'].mean()
    median_days = update_dates_pd['dias_desde_anterior'].median()
    
    print(f"\nIntervalo promedio entre actualizaciones: {avg_days:.1f} días")
    print(f"Intervalo mediano entre actualizaciones: {median_days:.0f} días")
    
    # Calcular próxima actualización esperada (usando mediana como mejor estimador)
    if last_update:
        next_update_avg = last_update + timedelta(days=avg_days)
        next_update_median = last_update + timedelta(days=median_days)
        
        print(f"\nPróxima actualización estimada (basada en promedio): {next_update_avg}")
        print(f"Próxima actualización estimada (basada en mediana): {next_update_median}")
        
        # Calcular día de la semana más común
        most_common_day = day_counts.orderBy(F.desc("count")).first()
        if most_common_day:
            print(f"\nDía de la semana más común para actualizaciones: {most_common_day['dia_semana']}")
        
        # Hora más común
        most_common_hour = hour_stats.first()
        if most_common_hour:
            print(f"Hora más común de actualización: {most_common_hour['hora_numero']}:00 hrs")
else:
    print("\nNo hay suficientes datos históricos para estimar la próxima actualización")